# 🧠 DSA Lab 11: Queues: Linear Queue, Circular Queue, Deque and Priority Queue
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 11 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: The Queue ADT

| Operation | Meaning | Time |
|---|---|---|
| `enqueue(x)` | Add x at the **rear** | O(1) |
| `dequeue()` | Remove and return the **front** element | O(1) |
| `front()` / `peek()` | Return the front element without removing it | O(1) |
| `rear()` | Return the last element without removing it | O(1) |
| `is_empty()` / `is_full()` | Check the state | O(1) |
| `size()` | Number of elements | O(1) |

**FIFO in action**

```text
enqueue(10)  enqueue(20)  enqueue(30)        dequeue() -> 10
front                 rear                   front        rear
  |                    |                       |            |
[10]  ->  [10, 20]  ->  [10, 20, 30]   ->    [20, 30]
```

### Why Not Just Use a Python List?

`list.append(x)` is O(1), but `list.pop(0)` must **shift all remaining elements left**, so it is **O(n)**. For a queue with a million items, every dequeue would move a million elements.

In [ ]:
import time
from collections import deque

n = 100_000
lst = list(range(n))
t = time.perf_counter()
while lst:
    lst.pop(0)                 # O(n) each
t_list = time.perf_counter() - t

dq = deque(range(n))
t = time.perf_counter()
while dq:
    dq.popleft()               # O(1) each
t_deque = time.perf_counter() - t
print(f"list.pop(0): {t_list:.3f}s   deque.popleft(): {t_deque:.4f}s")

## 🔄 Part B: Circular Queue (Array-Based)

With a fixed array, a simple queue moves `front` forward on every dequeue. Soon `rear` reaches the end of the array even though there is **empty space at the beginning**. A **circular queue** solves this by **wrapping around**: after the last index comes index 0. We use the **modulo** operator: `next = (i + 1) % capacity`.

**Wrap-around**

```text
capacity = 5, after enqueue A B C D, dequeue A B, enqueue E F

index:   0     1     2     3     4
       [ F ] [   ] [ C ] [ D ] [ E ]
         ^           ^
        rear        front          F wrapped around to index 0!
```

In [ ]:
class CircularQueue:
    def __init__(self, capacity):
        self.data = [None] * capacity
        self.capacity = capacity
        self.front = 0                     # index of the front element
        self.count = 0                     # number of elements

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.capacity

    def enqueue(self, x):
        if self.is_full():
            raise OverflowError("queue is full")
        rear = (self.front + self.count) % self.capacity
        self.data[rear] = x
        self.count += 1

    def dequeue(self):
        if self.is_empty():
            raise IndexError("queue is empty")
        x = self.data[self.front]
        self.data[self.front] = None
        self.front = (self.front + 1) % self.capacity
        self.count -= 1
        return x

    def peek(self):
        return None if self.is_empty() else self.data[self.front]

    def __str__(self):
        return str([self.data[(self.front + i) % self.capacity] for i in range(self.count)])

q = CircularQueue(5)
for x in "ABCD":
    q.enqueue(x)
q.dequeue(); q.dequeue()
q.enqueue("E"); q.enqueue("F")
print("queue:", q, "| raw array:", q.data, "| front index:", q.front)

> **💡 Why keep `count`?**
>
> With only `front` and `rear` indexes, an empty queue and a full queue look the same (front == rear). Keeping a `count` (or leaving one slot always empty) removes this ambiguity.

## 🔗 Part C: Linked-List Queue

Enqueue at the **tail** and dequeue from the **head** of a singly linked list. Both are O(1) and the queue never becomes full.

In [ ]:
class LinkedQueue:
    class _Node:
        def __init__(self, data):
            self.data, self.next = data, None

    def __init__(self):
        self.head = self.tail = None
        self.count = 0

    def enqueue(self, x):
        node = self._Node(x)
        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self.count += 1

    def dequeue(self):
        if self.head is None:
            raise IndexError("queue is empty")
        x = self.head.data
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        self.count -= 1
        return x

printer = LinkedQueue()
for job in ["CV.pdf", "Assignment.docx", "Timetable.png"]:
    printer.enqueue(job)
while printer.count:
    print("Printing", printer.dequeue())

## ↔️ Part D: Deque (Double-Ended Queue)

A **deque** (pronounced “deck”) allows insertion and deletion at **both ends** in O(1). It can act as both a stack and a queue. Python’s `collections.deque` is the standard tool and should be your default queue in Python.

| Operation | deque method | Time |
|---|---|---|
| Add at rear / front | `append(x)` / `appendleft(x)` | O(1) |
| Remove from rear / front | `pop()` / `popleft()` | O(1) |
| Look at ends | `dq[-1]` / `dq[0]` | O(1) |
| Fixed-size buffer | `deque(maxlen=k)` | Old items drop automatically |
| Rotate | `rotate(k)` | O(k) |

In [ ]:
from collections import deque

dq = deque()
dq.append("B"); dq.append("C"); dq.appendleft("A")
print(dq, dq.popleft(), dq.pop(), dq)

last_5 = deque(maxlen=5)                 # e.g. last 5 heart-rate readings
for bpm in [72, 75, 80, 78, 90, 95, 70]:
    last_5.append(bpm)
print("buffer:", list(last_5), "average:", sum(last_5) / len(last_5))

### Sliding Window Maximum with a Deque

**Problem:** for every window of k consecutive values, report the maximum. Keep a deque of **indexes** whose values are in **decreasing order**. The front is always the maximum of the current window. Each index enters and leaves once, so the total is **O(n)**.

In [ ]:
def window_max(nums, k):
    dq, result = deque(), []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:          # front index left the window
            dq.popleft()
        while dq and nums[dq[-1]] < x:     # smaller values can never be a maximum
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

## ⭐ Part E: Priority Queue (Preview)

In a **priority queue**, each element has a priority and the element with the **highest priority** (or lowest value) is served first, regardless of arrival time. It is usually implemented with a **heap** (Lab 15). Python provides `heapq`, a **min-heap**: the smallest item comes out first.

In [ ]:
import heapq

er = []                                   # emergency room: lower number = more serious
heapq.heappush(er, (3, "Ali - fever"))
heapq.heappush(er, (1, "Sara - accident"))
heapq.heappush(er, (2, "Usman - fracture"))
heapq.heappush(er, (3, "Zoya - cough"))
while er:
    priority, patient = heapq.heappop(er)
    print(f"Treating (priority {priority}): {patient}")

| Structure | Removes | Typical use |
|---|---|---|
| Stack | Newest item (LIFO) | Undo, recursion, DFS |
| Queue | Oldest item (FIFO) | Scheduling, buffering, BFS |
| Deque | Either end | Sliding windows, palindromes, both |
| Priority queue | Highest-priority item | Emergency rooms, Dijkstra, task schedulers |

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <queue>
#include <deque>
using namespace std;

class CircularQueue {
    int* data; int cap, front = 0, count = 0;
public:
    CircularQueue(int c) : cap(c) { data = new int[c]; }
    ~CircularQueue() { delete[] data; }
    bool enqueue(int x) {
        if (count == cap) return false;            // full
        data[(front + count) % cap] = x; count++; return true;
    }
    bool dequeue(int& out) {
        if (count == 0) return false;              // empty
        out = data[front]; front = (front + 1) % cap; count--; return true;
    }
};

int main() {
    queue<string> q;                 // std::queue: push, pop, front, back, empty
    q.push("A"); q.push("B");
    cout << q.front() << endl; q.pop();

    deque<int> d = {2, 3}; d.push_front(1); d.push_back(4);
    priority_queue<int> pq;          // MAX-heap by default in C++
    for (int x : {5, 1, 9}) pq.push(x);
    cout << pq.top() << endl;        // 9

    CircularQueue cq(3); int v;
    cq.enqueue(10); cq.enqueue(20); cq.dequeue(v); cout << v << endl;
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Circular Queue with Rear and Resize
Extend `CircularQueue` into `GrowingQueue` that adds a `rear()` method and, instead of raising an error when full, **doubles its capacity** (copy the elements in queue order starting at index 0).

```text
GrowingQueue(2): enqueue 1, 2, 3 → capacity 4, front 1, rear 3
```

In [ ]:
class GrowingQueue(CircularQueue):
    def rear(self):
        # TODO
        pass

    def enqueue(self, x):
        # TODO: if full, copy items in order into a double-size array, front = 0
        super().enqueue(x)

In [ ]:
# Test cell: run after your solution
g = GrowingQueue(2)
for x in [1, 2, 3]:
    g.enqueue(x)
assert g.capacity == 4 and g.peek() == 1 and g.rear() == 3
assert g.dequeue() == 1
for x in [4, 5, 6]:
    g.enqueue(x)
assert [g.dequeue() for _ in range(5)] == [2, 3, 4, 5, 6] and g.rear() is None
print("✅ Task 1 passed")

### Task 2 🟢 Time to Buy Tickets
People stand in a queue to buy cricket match tickets. `tickets[i]` is how many tickets person i wants; each person buys **one ticket per turn** (1 second) and then goes to the back of the line if they need more. Return the time taken for the person at position `k` to finish.

Simulate it with a `deque` of `[index, remaining]` pairs.

```text
tickets = [2, 3, 2], k = 2 → 6
```

In [ ]:
from collections import deque

def time_to_buy(tickets, k):
    q = deque([i, t] for i, t in enumerate(tickets))
    time = 0
    # TODO: serve the front person; re-queue if they still need tickets
    return time

In [ ]:
# Test cell: run after your solution
assert time_to_buy([2, 3, 2], 2) == 6
assert time_to_buy([5, 1, 1, 1], 0) == 8
assert time_to_buy([1], 0) == 1
print("✅ Task 2 passed")

### Task 3 🟡 Queue Using Two Stacks
Implement `StackQueue` with `enqueue`, `dequeue`, `peek` and `is_empty` using **only two Python lists used as stacks** (`append`/`pop()` only).

Hint: push into an `inbox` stack; when the `outbox` stack is empty, pour everything from inbox into outbox. Each element moves at most twice, so operations are **amortised O(1)**.

In [ ]:
class StackQueue:
    def __init__(self):
        self.inbox, self.outbox = [], []

    def enqueue(self, x):
        # TODO
        pass

    def dequeue(self):
        # TODO
        pass

    def peek(self):
        # TODO
        pass

    def is_empty(self):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
q = StackQueue()
q.enqueue(1); q.enqueue(2)
assert q.peek() == 1 and q.dequeue() == 1
q.enqueue(3)
assert q.dequeue() == 2 and q.dequeue() == 3 and q.is_empty()
try:
    q.dequeue(); raise AssertionError("should raise")
except IndexError:
    pass
print("✅ Task 3 passed")

### Task 4 🟡 Recent Requests Counter
A web server wants to know how many requests arrived in the **last 3000 milliseconds**. Implement `RecentCounter.ping(t)` which records a request at time `t` (strictly increasing) and returns the number of requests in `[t − 3000, t]`. Use a deque and drop old requests from the front.

```text
ping(1) → 1, ping(100) → 2, ping(3001) → 3, ping(3002) → 3
```

In [ ]:
from collections import deque

class RecentCounter:
    def __init__(self):
        self.q = deque()

    def ping(self, t):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
rc = RecentCounter()
assert [rc.ping(t) for t in [1, 100, 3001, 3002]] == [1, 2, 3, 3]
rc2 = RecentCounter(); assert rc2.ping(5000) == 1 and rc2.ping(9000) == 1
print("✅ Task 4 passed")

### Task 5 🟡 First Non-Repeating Character in a Stream
Characters of a chat message arrive one by one. After each character, report the **first character so far that has appeared exactly once** (or `#` if none). Use a queue of candidates and a count dictionary.

```text
"aabc" → "a#bb"
```

In [ ]:
from collections import deque

def first_unique_stream(stream):
    count, q, out = {}, deque(), []
    # TODO
    return "".join(out)

In [ ]:
# Test cell: run after your solution
assert first_unique_stream("aabc") == "a#bb"
assert first_unique_stream("zz") == "z#"
assert first_unique_stream("") == ""
assert first_unique_stream("abcabc") == "aaabc#"
print("✅ Task 5 passed")

### Task 6 🔴 Bank Counter Simulation
Simulate a bank with **one cashier**. Customers arrive at given minutes with given service times: `customers = [(arrival, service), ...]` sorted by arrival. The cashier serves in FIFO order.

Return `(average_wait, max_queue_length)`, where wait = start of service − arrival (rounded to 2 decimals) and the queue length counts customers **waiting** (not the one being served). Keep a queue of the service start times of waiting customers.

```text
[(0, 5), (1, 3), (2, 1), (10, 2)] → waits 0, 4, 6, 0 → (2.5, 2)
```

In [ ]:
from collections import deque

def simulate_bank(customers):
    if not customers:
        return 0.0, 0
    waiting = deque()            # service START times of customers still waiting
    free_at, waits, max_len = 0, [], 0
    # TODO: for each customer: remove waiting customers whose start time <= arrival,
    #       compute start = max(arrival, free_at), record the wait, update free_at,
    #       add to `waiting` if start > arrival, track the maximum queue length
    return round(sum(waits) / len(waits), 2), max_len

In [ ]:
# Test cell: run after your solution
assert simulate_bank([(0, 5), (1, 3), (2, 1), (10, 2)]) == (2.5, 2)
assert simulate_bank([(0, 1), (5, 1)]) == (0.0, 0)
assert simulate_bank([(0, 2), (0, 2), (0, 2)]) == (2.0, 2)
assert simulate_bank([]) == (0.0, 0)
print("✅ Task 6 passed")

### Task 7 🔴 Sliding Window Maximum (Traffic Monitoring)
A traffic camera records cars per minute. For each window of `k` minutes, report the **maximum**. Implement `window_max_k(counts, k)` in O(n) with a deque of indexes, and verify it against a brute-force version on random data.

```text
[1, 3, -1, -3, 5, 3, 6, 7], k = 3 → [3, 3, 5, 5, 6, 7]
```

In [ ]:
from collections import deque

def window_max_k(counts, k):
    dq, out = deque(), []
    # TODO: deque of indexes with decreasing values
    return out

In [ ]:
# Test cell: run after your solution
import random
assert window_max_k([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert window_max_k([4], 1) == [4]
for _ in range(30):
    r = [random.randint(0, 50) for _ in range(40)]; k = random.randint(1, 10)
    assert window_max_k(r, k) == [max(r[i:i + k]) for i in range(len(r) - k + 1)]
print("✅ Task 7 passed")

## 🔬 Bonus: Breadth-First Search Preview (Shortest Path on a Grid)

In [ ]:
# Bonus: run after completing the tasks
from collections import deque

campus = ["S..#....",
          ".#.#.##.",
          ".#...#..",
          ".####.#.",
          "......#E"]
rows, cols = len(campus), len(campus[0])
dist = {(0, 0): 0}
q = deque([(0, 0)])
while q:                                   # BFS explores level by level using a QUEUE
    r, c = q.popleft()
    for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and campus[nr][nc] != "#" and (nr, nc) not in dist:
            dist[(nr, nc)] = dist[(r, c)] + 1
            q.append((nr, nc))
print("Shortest walk from S to E:", dist.get((rows - 1, cols - 1)), "steps")

---
## 🎉 Lab 11 Complete!

Next: **Lab 12: Hashing and Hash Tables**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab11/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)